# Capstone - Content Refresh Prioritization

> **Author:** Shanky Pal  
> **Lane:** Lane 2 - Refresh / Content Opportunity Scoring  
> **Repo:** [github.com/Shanky085/day1](https://github.com/Shanky085/day1)  
> **Date:** August 2026

## 1. Question

**Research question:** Can observed content and search-performance signals be used to prioritize pages for human review for a possible content refresh?

**Decision supported:** Which pages should a content or SEO reviewer examine first?

**Action:** A human reviewer inspects the recommended pages and decides whether to refresh, monitor, or leave each one.

**Cost of a wrong call:** Valuable content may be ignored while resources are spent on pages that do not need attention, reducing the effectiveness of the content optimization process.

The output is a prioritization aid. It does not claim that a refresh will cause improved search performance.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique clients:", df["client_id"].nunique())

Rows: 30000
Columns: 44
Unique clients: 32


## 2. Data

30,000 page-level observations. Features: search volume, CPC, competition level, word/char count, impressions/clicks/pageviews/sessions (90d), content age, CTR.

**Label:** `trend_direction == "down"` (binary, ~54.2% positive class).

**Leakage exclusions:** `trend_direction`, `trend_pct`, `content_id`, `client_id` are never features.

In [2]:
print("Target:")
print(df["trend_direction"].value_counts())
df["needs_attention"] = (df["trend_direction"] == "down").astype(int)
print("\nBinary target:")
print(df["needs_attention"].value_counts())

Target:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Binary target:
needs_attention
1    16262
0    13738
Name: count, dtype: int64


## 3. Methodology

**Target:** `trend_direction == "down"` (binary).  
**Model:** Logistic Regression (sklearn Pipeline with ColumnTransformer).  
**Split:** `GroupShuffleSplit` by `client_id` (80/20, zero client overlap).  
**Baseline:** 3-point rule (stale + low CTR + visible), thresholds from dataset medians.  

**Validation:** Grouped split (honest) vs random split (inflated) to quantify data leakage from same-client rows.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

features_num = [
    "search_volume", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "content_age_days", "ctr"
]
features_cat = ["competition_level"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, df["needs_attention"], df["client_id"]))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), features_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), features_cat)
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

X_train = train[features_num + features_cat]
y_train = train["needs_attention"]
X_test = test[features_num + features_cat]
y_test = test["needs_attention"]

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
print("Train:", len(train), "| Test:", len(test))
print("Train clients:", train["client_id"].nunique(),
      "| Test clients:", test["client_id"].nunique())
print("Client overlap:", len(set(train["client_id"]) & set(test["client_id"])))

Train: 23837 | Test: 6163
Train clients: 25 | Test clients: 7
Client overlap: 0


In [4]:
# Baseline: 3-point scoring rule (thresholds from full-dataset medians)
ac = df["content_age_days"].median()
cc = df["ctr"].median()
ic = df["impressions_90d"].median()

baseline_score = (
    (test["content_age_days"] >= ac).astype(int)
    + (test["ctr"] < cc).astype(int)
    + (test["impressions_90d"] >= ic).astype(int)
)

print("Baseline median thresholds:")
print(f"  content_age_days >= {ac:.0f}")
print(f"  ctr < {cc:.6f}")
print(f"  impressions_90d >= {ic:.0f}")

Baseline median thresholds:
  content_age_days >= 236
  ctr < 0.070000
  impressions_90d >= 731


In [5]:
from sklearn.metrics import average_precision_score, roc_auc_score

ap_model = average_precision_score(y_test, y_prob)
auc_model = roc_auc_score(y_test, y_prob)

bs_prob = baseline_score.astype(float) / 3.0
ap_baseline = average_precision_score(y_test, bs_prob)
auc_baseline = roc_auc_score(y_test, bs_prob)

print(f"Logistic Regression - AP: {ap_model:.4f}, ROC-AUC: {auc_model:.4f}")
print(f"W04 baseline rule    - AP: {ap_baseline:.4f}, ROC-AUC: {auc_baseline:.4f}")
print(f"Delta (model - baseline) - AP: +{ap_model - ap_baseline:.4f}, ROC-AUC: +{auc_model - auc_baseline:.4f}")

Logistic Regression - AP: 0.5582, ROC-AUC: 0.5621
W04 baseline rule    - AP: 0.4974, ROC-AUC: 0.4644
Delta (model - baseline) - AP: +0.0609, ROC-AUC: +0.0977


## 4. Results

On the grouped test split, Logistic Regression achieves moderate improvement over the baseline rule on both Average Precision and ROC-AUC.

| Method | Average Precision | ROC-AUC |
|---|---|---|
| W04 baseline rule | shown above | shown above |
| Logistic Regression | shown above | shown above |

In [6]:
r = pd.DataFrame({
    "method": ["W04 baseline", "Logistic Regression"],
    "avg_precision": [ap_baseline, ap_model],
    "roc_auc": [auc_baseline, auc_model]
})
r["delta_ap"] = r["avg_precision"] - r["avg_precision"].iloc[0]
r["delta_auc"] = r["roc_auc"] - r["roc_auc"].iloc[0]
print(r.to_string(index=False))

             method  avg_precision  roc_auc  delta_ap  delta_auc
       W04 baseline       0.497362 0.464380  0.000000   0.000000
Logistic Regression       0.558214 0.562052  0.060852   0.097671


In [7]:
from sklearn.model_selection import train_test_split

xr, xv, yr, yv = train_test_split(
    df[features_num + features_cat],
    df["needs_attention"],
    test_size=0.2,
    random_state=42,
    stratify=df["needs_attention"]
)

m2 = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
m2.fit(xr, yr)

ap_r = average_precision_score(yv, m2.predict_proba(xv)[:, 1])
auc_r = roc_auc_score(yv, m2.predict_proba(xv)[:, 1])

print(f"Random split   - AP: {ap_r:.4f}, ROC-AUC: {auc_r:.4f}")
print(f"Grouped split  - AP: {ap_model:.4f}, ROC-AUC: {auc_model:.4f}")
print(f"Inflation      - AP: +{ap_r - ap_model:.4f}, ROC-AUC: +{auc_r - auc_model:.4f}")
print("Client overlap in grouped split:", len(set(train["client_id"]) & set(test["client_id"])))

Random split   - AP: 0.6251, ROC-AUC: 0.6111
Grouped split  - AP: 0.5582, ROC-AUC: 0.5621
Inflation      - AP: +0.0669, ROC-AUC: +0.0490
Client overlap in grouped split: 0


## 5. Limitations

1. **Proxy target:** same-window proxy, not forward-looking decline. The label `trend_direction` is computed from the same observation window as the features. No temporal separation between features and label.
2. **No time-aware split:** the grouped split prevents same-client leakage but not temporal leakage. GUIDE.md recommends features from prior 90 days predicting decline over next 30 days; this analysis uses a single snapshot.
3. **No Precision@K:** GUIDE.md specifies Precision@K for review queues. This capstone reports AP and ROC-AUC; Precision@K should be added in a production deployment.
4. **Single seed:** no confidence intervals or stability checks across multiple random seeds.
5. **Moderate AUC:** ~0.56 ROC-AUC suggests the model provides weak but real signal. Results are directional, not production-ready.

**Honest claims:** observed, measured, directional, decision-support. Not causal. Does not predict search engine behavior.

## 6. Ranked recommendations

| Priority | Action | Reason code | Human review required |
|---|---|---|---|
| 1 | Refresh candidate | stale_visible_page | Yes |
| 2 | Monitor closely | low_ctr_visible_page | Yes |
| 3 | Periodic review | stale_content | Yes |
| 4 | Leave as-is | other | No |

**No-go automation:** The model output is a priority score. All refresh/monitor/prune decisions require human review. The system recommends; a person decides.

In [8]:
w = df.copy()
w["s"] = w["content_age_days"] >= ac
w["l"] = w["ctr"] < cc
w["v"] = w["impressions_90d"] >= ic
w["score"] = w["s"].astype(int) + w["l"].astype(int) + w["v"].astype(int)
w["action"] = w["score"].map({3: "refresh", 2: "monitor"}).fillna("leave")
w["reason"] = np.select(
    [w["s"] & w["v"], w["l"] & w["v"], w["s"] & ~w["v"]],
    ["stale_visible_page", "low_ctr_visible_page", "stale_content"],
    default="other"
)
print(w["action"].value_counts())
print("\nReason code distribution:")
print(w["reason"].value_counts())

action
leave      15112
monitor    13001
refresh     1887
Name: count, dtype: int64

Reason code distribution:
reason
other                   13365
stale_visible_page       7714
stale_content            7496
low_ctr_visible_page     1425
Name: count, dtype: int64


## 7. Artifacts

Aggregate metrics and public-safe charts. No client names, URLs, queries, or credentials.

In [9]:
import os

od = Path("../../work/outputs")
od.mkdir(parents=True, exist_ok=True)
r.to_csv(od / "model_vs_baseline_metrics.csv", index=False)
print("Saved:", od / "model_vs_baseline_metrics.csv")

figures_dir = Path("../../work/figures")
print("Existing figures:")
if figures_dir.exists():
    for f in sorted(figures_dir.glob("*.png")):
        print(" ", f.name)
else:
    print(" (none found)")

Saved: ..\..\work\outputs\model_vs_baseline_metrics.csv
Existing figures:
  model_vs_baseline_average_precision.png
  model_vs_baseline_roc_auc.png


In [10]:
print("Reproducibility")
print("=" * 50)
print("Python: 3.11 | sklearn | pandas | numpy")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
print(f"Train: {len(train)} | Test: {len(test)}")
print(f"Train clients: {train['client_id'].nunique()} | Test clients: {test['client_id'].nunique()}")
print(f"Model AP: {ap_model:.4f} | Model AUC: {auc_model:.4f}")
print(f"Baseline AP: {ap_baseline:.4f} | Baseline AUC: {auc_baseline:.4f}")
print(f"Random-split AP: {ap_r:.4f} | Grouped-split AP: {ap_model:.4f}")

Reproducibility
Python: 3.11 | sklearn | pandas | numpy
Rows: 30000 | Columns: 45
Train: 23837 | Test: 6163
Train clients: 25 | Test clients: 7
Model AP: 0.5582 | Model AUC: 0.5621
Baseline AP: 0.4974 | Baseline AUC: 0.4644
Random-split AP: 0.6251 | Grouped-split AP: 0.5582


## Self-check

- [x] All cells run without errors
- [x] No client names, URLs, queries, or credentials printed
- [x] All claims are observed/measured/directional/decision-support
- [x] Model vs baseline comparison uses the same test set and metrics
- [x] Leakage audit passed (no `trend_direction` or `trend_pct` in features)
- [x] Grouped split confirms zero client overlap between train and test
- [x] Action playbook requires human review for all recommendations